# 03. 피처 엔지니어링

**목적**: `02_eda`에서 확인한 물리적 관계(풍속 3제곱 법칙, 풍향 영향, 계절×풍속 비선형 상호작용, 공기밀도, LDAPS의 지형 해상도 우위)를 근거로 모델이 학습할 입력 변수(피처)를 만든다.

- **입력**: `data/processed/train_base_wide.parquet`, `test_base_wide.parquet` (`01_preprocessing` 산출물, 격자별 컬럼 보존)
- **출력**: `data/processed/train_features_v1.parquet`, `test_features_v1.parquet`

**피처 그룹** (근거는 `wind-domain-features` 스킬, 세부 수치 근거는 `reports/02_eda.md`)

| 그룹 | 내용 |
|---|---|
| A | 데이터 품질 수정 (LDAPS 결측 3시각 보간, 습도 100% 클리핑) |
| B | 그룹가중 풍속/풍향 (GFS 10/80/100m·850hPa, LDAPS 10m) |
| C | 윈드시어 지수(α) |
| D | 공기밀도·밀도보정 풍속 |
| E | 대기 안정도·난류 대리변수 |
| F | 돌풍/변동성 |
| G | NWP 앙상블 불일치(GFS-LDAPS 차이) |
| H | 발표분 내 변화율 |
| I | 시간(달력) 피처 |

**leakage-guard 원칙**: 모든 피처는 train/test 동일한 함수로 생성하고, 같은 발표분(`data_available_kst_dtm`) 경계를 넘는 정보(다른 발표분, 미래 실측치)는 사용하지 않는다. SCADA는 피처로 쓰지 않는다(이 노트북에서 학습되는 값이 아니라 test에 없는 원천이기 때문).


## 0. 셋업

패키지를 불러오고, `01_preprocessing`이 만든 wide 캐시(격자별 컬럼이 살아있는 버전)를 불러옵니다. 그룹가중 풍속 계산에는 격자별 컬럼이 꼭 필요해서 `agg`가 아니라 `wide`를 씁니다.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

# 이 노트북이 어디서 실행되든 REPO_ROOT를 찾아낸다
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

sys.path.insert(0, str(REPO_ROOT))
from src.metric import TARGET_COLS, CAPACITY_KWH

PROCESSED_DIR = REPO_ROOT / "data" / "processed"

print("python executable:", sys.executable)
print("REPO_ROOT:", REPO_ROOT)
print("TARGET_COLS:", TARGET_COLS)
print("CAPACITY_KWH:", CAPACITY_KWH)


python executable: d:\공모전\wind_forecast\venv\Scripts\python.exe
REPO_ROOT: d:\공모전\wind_forecast
TARGET_COLS: ['kpx_group_1', 'kpx_group_2', 'kpx_group_3']
CAPACITY_KWH: {'kpx_group_1': 21600, 'kpx_group_2': 21600, 'kpx_group_3': 21000}


In [2]:
train_base_wide = pd.read_parquet(PROCESSED_DIR / "train_base_wide.parquet")
test_base_wide = pd.read_parquet(PROCESSED_DIR / "test_base_wide.parquet")

print("train_base_wide:", train_base_wide.shape)
print("test_base_wide:", test_base_wide.shape)


train_base_wide: (26304, 801)
test_base_wide: (8760, 799)


### 0-1. 그룹-격자 가중치 (`02_eda`에서 확정된 값 그대로 사용)

`02_eda.ipynb`에서 `info.xlsx`의 터빈별 좌표로 각 터빈의 최근접 격자를 구하고, 그룹 안에서 터빈이 어느 격자에 몇 대씩 배정되는지 비율로 가중치를 만들었습니다(예: group_2는 LDAPS 격자 두 개에 정확히 반반 갈림). 이 가중치는 터빈 위치(고정된 물리적 좌표)에서만 나온 값이라 다시 계산해도 항상 같은 결과가 나옵니다 — 그래서 여기서는 `02_eda`가 만든 값을 그대로 상수로 가져다 씁니다(중복 계산 대신 재사용, 값 출처는 `reports/02_eda.md` 2-1절).


In [3]:
# 02_eda.ipynb 2-1절에서 계산한 값 그대로 (info.xlsx 터빈 좌표 -> 최근접 격자 비율)
GROUP_GRID_WEIGHTS = {
    "kpx_group_1": {
        "gfs": {5: 1.0},
        "ldaps": {5: 2 / 3, 6: 1 / 3},
    },
    "kpx_group_2": {
        "gfs": {5: 1.0},
        "ldaps": {6: 0.5, 11: 0.5},
    },
    "kpx_group_3": {
        "gfs": {5: 1.0},
        "ldaps": {12: 0.8, 6: 0.2},
    },
}

for group, sources in GROUP_GRID_WEIGHTS.items():
    for source, weights in sources.items():
        assert abs(sum(weights.values()) - 1.0) < 1e-9, f"{group}/{source} 가중치 합이 1이 아닙니다"

GROUP_GRID_WEIGHTS


{'kpx_group_1': {'gfs': {5: 1.0},
  'ldaps': {5: 0.6666666666666666, 6: 0.3333333333333333}},
 'kpx_group_2': {'gfs': {5: 1.0}, 'ldaps': {6: 0.5, 11: 0.5}},
 'kpx_group_3': {'gfs': {5: 1.0}, 'ldaps': {12: 0.8, 6: 0.2}}}

### 0-2. 공통 헬퍼 함수

- `group_weighted_uv`: 바람의 동서(u)·남북(v) 성분을 그룹 가중치로 섞음(풍향은 각도라서 평균 내기 전에 벡터 성분을 먼저 섞어야 방향이 왜곡되지 않습니다 — `wind-domain-features` 2절)
- `group_weighted_scalar`: 기압·기온처럼 방향이 없는 값(스칼라)을 그룹 가중치로 섞음
- `wind_speed`, `wind_direction`: u/v 성분을 풍속(m/s)·풍향(도, 기상학 관례: 바람이 불어오는 방향)으로 변환


In [4]:
def group_weighted_uv(df, group, source, u_var, v_var):
    """그룹에 속한 터빈들의 실제 최근접 격자 비율(GROUP_GRID_WEIGHTS)로 u/v 성분을 가중평균한다."""
    weights = GROUP_GRID_WEIGHTS[group][source]
    u_total, v_total = 0.0, 0.0
    for grid_id, weight in weights.items():
        u_total = u_total + df[f"{source}_g{grid_id}_{u_var}"] * weight
        v_total = v_total + df[f"{source}_g{grid_id}_{v_var}"] * weight
    return u_total, v_total


def group_weighted_scalar(df, group, source, var):
    """방향이 없는 값(기압, 기온 등)을 그룹 가중치로 가중평균한다."""
    weights = GROUP_GRID_WEIGHTS[group][source]
    total = 0.0
    for grid_id, weight in weights.items():
        total = total + df[f"{source}_g{grid_id}_{var}"] * weight
    return total


def wind_speed(u, v):
    return np.sqrt(u ** 2 + v ** 2)


def wind_direction(u, v):
    """기상학 관례: 바람이 '불어오는' 방향(도, 0=북, 90=동)"""
    return (270 - np.degrees(np.arctan2(v, u))) % 360


## A. 데이터 품질 수정

앞서 확인하고 결정한 두 가지를 여기서 적용합니다(민석님과 논의 후 확정한 방침).

1. **LDAPS 상대습도 100% 초과 클리핑**: 상대습도는 물리적으로 100%를 넘을 수 없으므로, 100을 넘는 값은 100으로 잘라냅니다.
2. **test LDAPS 결측 3개 시각 보간**: `2025-04-08 17:00`/`2025-06-18 18:00`/`2025-07-18 06:00` 3개 시각에서 16개 격자 전부의 보조 변수(구름량·경계층고도·기압·2m 기온/습도 등)가 동시에 빠져 있었습니다. 핵심 피처인 10m 풍속(u/v)은 이 세 시각 모두 멀쩡해서 실질 영향은 작지만, 보조 변수를 피처로 쓸 경우를 대비해 **같은 발표분(`ldaps_data_available_kst_dtm`) 안의 다른 시각 값으로 선형보간**합니다. 발표분 하나가 24시간을 통째로 담당하므로, 이 보간은 그 발표분이 공개된 시점에 이미 다 알 수 있는 정보만 사용하는 셈이라 `leakage-guard` 기준으로 안전합니다.

train과 test에 **동일한 함수**를 적용합니다(전처리 원칙).


In [5]:
def fix_ldaps_quality(df):
    """LDAPS 상대습도 100% 클리핑 + 같은 발표분 내 결측 시간보간. train/test 동일 적용."""
    df = df.sort_values("forecast_kst_dtm").reset_index(drop=True).copy()

    # 1) 상대습도(heightAboveGround_2_r) 100% 클리핑
    rh_cols = [c for c in df.columns if c.startswith("ldaps_g") and c.endswith("heightAboveGround_2_r")]
    df[rh_cols] = df[rh_cols].clip(upper=100)

    # 2) 같은 발표분 내 시간 선형보간 (발표분 경계를 넘지 않도록 groupby 안에서만 보간)
    ldaps_cols = [c for c in df.columns if c.startswith("ldaps_g")]
    if df[ldaps_cols].isna().any().any():
        df[ldaps_cols] = df.groupby("ldaps_data_available_kst_dtm")[ldaps_cols].transform(
            lambda s: s.interpolate(method="linear", limit_area="inside")
        )
    return df


print("클리핑 전 LDAPS 습도 최댓값 (train):", train_base_wide.filter(regex="heightAboveGround_2_r$").filter(regex="^ldaps").max().max())
print("클리핑 전 LDAPS 습도 최댓값 (test):", test_base_wide.filter(regex="heightAboveGround_2_r$").filter(regex="^ldaps").max().max())
print("보간 전 test LDAPS 결측 총 개수:", test_base_wide.filter(regex="^ldaps_g").isna().sum().sum())

train_clean = fix_ldaps_quality(train_base_wide)
test_clean = fix_ldaps_quality(test_base_wide)

print("클리핑 후 LDAPS 습도 최댓값 (train):", train_clean.filter(regex="heightAboveGround_2_r$").filter(regex="^ldaps").max().max())
print("클리핑 후 LDAPS 습도 최댓값 (test):", test_clean.filter(regex="heightAboveGround_2_r$").filter(regex="^ldaps").max().max())
print("보간 후 test LDAPS 결측 총 개수:", test_clean.filter(regex="^ldaps_g").isna().sum().sum())
print("train shape:", train_clean.shape, "/ test shape:", test_clean.shape)


클리핑 전 LDAPS 습도 최댓값 (train): 110.38448
클리핑 전 LDAPS 습도 최댓값 (test): 113.077354
보간 전 test LDAPS 결측 총 개수: 752
클리핑 후 LDAPS 습도 최댓값 (train): 100.0
클리핑 후 LDAPS 습도 최댓값 (test): 100.0
보간 후 test LDAPS 결측 총 개수: 0
train shape: (26304, 801) / test shape: (8760, 799)


## B. 핵심 풍속/풍향 피처

발전량 공식 `P = ½·ρ·A·Cp·v³`에서 풍속(v)이 세제곱으로 들어가므로 풍속이 압도적 1순위 피처입니다(`wind-domain-features` 1절). u/v 성분을 풍속·풍향으로 바꾸고, 풍향은 원형 변수라 그대로 쓰면 359°와 1°가 멀어 보이므로 `sin`/`cos`로 인코딩합니다.

- **GFS**: 전 그룹이 격자 5 하나로 수렴하므로(`GROUP_GRID_WEIGHTS`), 그룹 구분 없이 공유 피처로 한 번만 계산합니다. 10/80/100m와 850hPa(산 정상부 자유대기, 태백 가덕산은 해발 ~1,000m)를 모두 만들고, 허브높이(117m)에 가장 가까운 100m에 세제곱까지 적용합니다.
- **LDAPS**: 그룹 안에서 터빈이 격자별로 갈리므로(예: group_2는 정확히 반반) 그룹마다 따로 계산합니다. 10m만 제공되지만, EDA에서 LDAPS 10m이 GFS 전 높이보다 상관이 높았으므로(지형 해상도 우위) 세제곱까지 적용합니다.


In [6]:
GFS_HEIGHT_VARS = {
    "gfs_ws10m": ("heightAboveGround_10_10u", "heightAboveGround_10_10v"),
    "gfs_ws80m": ("heightAboveGround_80_u", "heightAboveGround_80_v"),
    "gfs_ws100m": ("heightAboveGround_100_100u", "heightAboveGround_100_100v"),
    "gfs_ws850hpa": ("isobaricInhPa_850_u", "isobaricInhPa_850_v"),
}


def build_core_wind_features(df):
    feats = pd.DataFrame(index=df.index)
    if "forecast_id" in df.columns:  # train_base_wide에는 없고 test_base_wide에만 있음(제출용 식별자)
        feats["forecast_id"] = df["forecast_id"]
    feats["forecast_kst_dtm"] = df["forecast_kst_dtm"]

    # GFS: 전 그룹 공유(격자 5 하나, 가중치 1.0) -> 아무 그룹이나 대표로 사용
    for name, (u_var, v_var) in GFS_HEIGHT_VARS.items():
        u, v = group_weighted_uv(df, "kpx_group_1", "gfs", u_var, v_var)
        feats[name] = wind_speed(u, v)
        if name == "gfs_ws100m":
            wd = wind_direction(u, v)
            feats["gfs_wd100m_sin"] = np.sin(np.radians(wd))
            feats["gfs_wd100m_cos"] = np.cos(np.radians(wd))
            feats["gfs_ws100m_sq"] = feats[name] ** 2
            feats["gfs_ws100m_cube"] = feats[name] ** 3

    # LDAPS: 그룹별 가중치가 다름 -> 그룹마다 계산
    for group in TARGET_COLS:
        u, v = group_weighted_uv(df, group, "ldaps", "heightAboveGround_10_10u", "heightAboveGround_10_10v")
        ws = wind_speed(u, v)
        wd = wind_direction(u, v)
        feats[f"{group}_ldaps_ws10m"] = ws
        feats[f"{group}_ldaps_ws10m_sq"] = ws ** 2
        feats[f"{group}_ldaps_ws10m_cube"] = ws ** 3
        feats[f"{group}_ldaps_wd10m_sin"] = np.sin(np.radians(wd))
        feats[f"{group}_ldaps_wd10m_cos"] = np.cos(np.radians(wd))

    return feats


train_feats = build_core_wind_features(train_clean)
test_feats = build_core_wind_features(test_clean)

print("train_feats:", train_feats.shape)
display(train_feats.head(3))
print("결측 총량:", train_feats.isna().sum().sum())


train_feats: (26304, 24)


,forecast_kst_dtm,gfs_ws10m,gfs_ws80m,gfs_ws100m,gfs_wd100m_sin,gfs_wd100m_cos,gfs_ws100m_sq,gfs_ws100m_cube,gfs_ws850hpa,kpx_group_1_ldaps_ws10m,...,kpx_group_2_ldaps_ws10m,kpx_group_2_ldaps_ws10m_sq,kpx_group_2_ldaps_ws10m_cube,kpx_group_2_ldaps_wd10m_sin,kpx_group_2_ldaps_wd10m_cos,kpx_group_3_ldaps_ws10m,kpx_group_3_ldaps_ws10m_sq,kpx_group_3_ldaps_ws10m_cube,kpx_group_3_ldaps_wd10m_sin,kpx_group_3_ldaps_wd10m_cos
0,2022-01-01 01:00:00,2.516111,3.159457,3.176957,-0.985030,-0.172384,10.093054,32.065196,4.717845,5.341039,...,6.119808,37.452050,229.199360,-0.999299,-0.037441,6.526803,42.599162,278.036357,-0.999520,0.030975
1,2022-01-01 02:00:00,2.473222,3.056382,3.076688,-0.978158,-0.207862,9.466008,29.123953,4.277518,4.534232,...,5.227108,27.322654,142.818449,-0.999971,-0.007594,5.408459,29.251431,158.205175,-0.993959,0.109755
2,2022-01-01 03:00:00,2.507237,3.105247,3.131877,-0.961744,-0.273951,9.808655,30.719503,4.022996,4.326244,...,4.969057,24.691523,122.693573,-0.999158,-0.041024,5.225686,27.307797,142.701980,-0.995810,0.091450


결측 총량: 0


## C. 윈드시어 지수(α)

윈드시어 멱법칙 `v(z) = v(ref)·(z/z_ref)^α`에서 높이 간 풍속비로 시어 지수 α를 역산합니다. α가 크면 높이에 따라 풍속 차이가 크다는 뜻으로, 대기가 안정적인 상태(예: 야간 지표 냉각으로 생기는 역전층)의 대리변수입니다(`wind-domain-features` 3절). GFS는 10m/100m를 모두 제공하므로 이 둘로 계산합니다(전 그룹 공유).

풍속이 0에 가까우면 로그 계산이 불안정해지므로, 아주 작은 하한(0.1 m/s)을 두어 나눗셈·로그 오류를 막습니다.


In [7]:
WS_FLOOR = 0.1  # log/나눗셈 안정화용 하한(m/s)

def add_wind_shear(feats):
    ws10 = feats["gfs_ws10m"].clip(lower=WS_FLOOR)
    ws100 = feats["gfs_ws100m"].clip(lower=WS_FLOOR)
    feats["gfs_shear_alpha"] = np.log(ws100 / ws10) / np.log(100 / 10)
    return feats


train_feats = add_wind_shear(train_feats)
test_feats = add_wind_shear(test_feats)

print(train_feats["gfs_shear_alpha"].describe())


count    26304.000000
mean         0.165606
std          0.120548
min         -1.056525
25%          0.102609
50%          0.174430
75%          0.238552
max          1.101015
Name: gfs_shear_alpha, dtype: float64


## D. 공기밀도 보정

P ∝ ρ이고 ρ = p/(R·T)입니다(이상기체 법칙, R=287.05). 겨울(저온·고압)에는 여름보다 밀도가 10% 이상 높아, **같은 풍속이라도 겨울에 발전량이 더 큽니다** — EDA에서 확인한 "계절×풍속 비선형 상호작용"(저·중풍속 구간에서 겨울이 유리)의 물리적 원인이 바로 이것입니다. IEC 61400-12-1 표준 방식으로 밀도보정 풍속(`ws·(ρ/1.225)^(1/3)`, 1.225는 표준대기 밀도)도 함께 만듭니다.

데이터는 지표기압(Pa)·기온(켈빈)으로 확인했으므로(직접 조회함) 단위 변환 없이 그대로 공식에 넣습니다. GFS는 그룹 공유, LDAPS는 그룹별 가중치가 다르므로 그룹마다 계산합니다.


In [8]:
R_DRY_AIR = 287.05  # J/(kg*K), 건조공기 기체상수
RHO_STANDARD = 1.225  # kg/m^3, IEC 표준대기 밀도


def air_density(sp_pa, t_kelvin):
    return sp_pa / (R_DRY_AIR * t_kelvin)


def add_air_density(df, feats):
    # GFS: 전 그룹 공유(격자 5 하나)
    gfs_sp = group_weighted_scalar(df, "kpx_group_1", "gfs", "surface_0_sp")
    gfs_t = group_weighted_scalar(df, "kpx_group_1", "gfs", "heightAboveGround_2_2t")
    feats["gfs_rho"] = air_density(gfs_sp, gfs_t)
    feats["gfs_ws100m_rho_corrected"] = feats["gfs_ws100m"] * (feats["gfs_rho"] / RHO_STANDARD) ** (1 / 3)

    # LDAPS: 그룹별 가중치
    for group in TARGET_COLS:
        ldaps_sp = group_weighted_scalar(df, group, "ldaps", "surface_0_sp")
        ldaps_t = group_weighted_scalar(df, group, "ldaps", "heightAboveGround_2_t")
        rho = air_density(ldaps_sp, ldaps_t)
        feats[f"{group}_ldaps_rho"] = rho
        feats[f"{group}_ldaps_ws10m_rho_corrected"] = (
            feats[f"{group}_ldaps_ws10m"] * (rho / RHO_STANDARD) ** (1 / 3)
        )
    return feats


train_feats = add_air_density(train_clean, train_feats)
test_feats = add_air_density(test_clean, test_feats)

print(train_feats[["gfs_rho", "gfs_ws100m_rho_corrected"]].describe())


            gfs_rho  gfs_ws100m_rho_corrected
count  26304.000000              26304.000000
mean       1.147220                  3.885988
std        0.046683                  3.109855
min        1.044308                  0.001425
25%        1.106282                  1.859625
50%        1.144115                  2.938454
75%        1.185206                  4.783812
max        1.285874                 26.004962


## E. 대기 안정도·난류 대리변수

대기가 불안정하면(지표가 상층보다 많이 따뜻하면) 상층의 강한 바람이 지표로 잘 섞여 내려와 지상 풍속이 세지는 경향이 있습니다(`wind-domain-features` 5절). 안정도를 직접 재는 변수는 없지만 아래 세 가지로 대리(proxy)할 수 있습니다.

- **2m-850hPa 기온차(GFS)**: 지표가 상층보다 얼마나 따뜻한지 → 클수록 불안정·혼합 활발
- **경계층고도(LDAPS `blh`)**: 지표 근처 공기가 섞이는 층의 두께. LDAPS가 직접 제공하는 값이라 대리변수를 따로 계산할 필요 없이 그대로 그룹가중 평균만 하면 됨
- **50m 최대-최소 풍속차(LDAPS)**: 같은 시각·같은 높이에서 바람이 얼마나 출렁였는지 → 난류 강도 대리

**정확히는**: `50MUmax`/`50MVmax`는 그 시간 동안의 U 최댓값과 V 최댓값을 각각 따로 기록한 값이라, 두 값이 실제로 "같은 순간"에 함께 발생했다는 보장이 없습니다(최소값도 마찬가지). 그래서 `sqrt(Umax²+Vmax²)`은 진짜 순간 최대풍속이 아니라 근사치이고, 이 때문에 "최대-최소" 차이가 드물게 음수로 나올 수 있습니다(실행 결과 확인됨). 그래도 난류가 강할수록 이 값이 커지는 경향 자체는 유지되어 대리변수로는 유효하다고 판단해 그대로 사용합니다.


In [9]:
def add_stability_turbulence(df, feats):
    # GFS: 2m-850hPa 기온차 (전 그룹 공유)
    gfs_t2m = group_weighted_scalar(df, "kpx_group_1", "gfs", "heightAboveGround_2_2t")
    gfs_t850 = group_weighted_scalar(df, "kpx_group_1", "gfs", "isobaricInhPa_850_t")
    feats["gfs_temp_diff_2m_850hpa"] = gfs_t2m - gfs_t850

    for group in TARGET_COLS:
        # LDAPS 경계층고도
        feats[f"{group}_ldaps_blh"] = group_weighted_scalar(df, group, "ldaps", "etc_0_blh")

        # LDAPS 50m 돌풍 범위: 최대/최소 U·V를 각각 그룹가중 후 풍속으로 변환해 차이를 냄
        u_max = group_weighted_scalar(df, group, "ldaps", "heightAboveGround_50_50MUmax")
        v_max = group_weighted_scalar(df, group, "ldaps", "heightAboveGround_50_50MVmax")
        u_min = group_weighted_scalar(df, group, "ldaps", "heightAboveGround_50_50MUmin")
        v_min = group_weighted_scalar(df, group, "ldaps", "heightAboveGround_50_50MVmin")
        gust_speed_max = wind_speed(u_max, v_max)
        gust_speed_min = wind_speed(u_min, v_min)
        feats[f"{group}_ldaps_gust_range_50m"] = gust_speed_max - gust_speed_min
    return feats


train_feats = add_stability_turbulence(train_clean, train_feats)
test_feats = add_stability_turbulence(test_clean, test_feats)

print(train_feats[["gfs_temp_diff_2m_850hpa", "kpx_group_1_ldaps_blh", "kpx_group_1_ldaps_gust_range_50m"]].describe())


       gfs_temp_diff_2m_850hpa  kpx_group_1_ldaps_blh  \
count             26304.000000           26304.000000   
mean                  3.215940             342.234495   
std                   3.515691             289.868157   
min                  -8.205100              22.797594   
25%                   0.622558             159.376367   
50%                   3.228735             221.874927   
75%                   5.901412             435.630185   
max                  10.628670            3009.745667   

       kpx_group_1_ldaps_gust_range_50m  
count                      26304.000000  
mean                           0.516111  
std                            1.358083  
min                          -10.840313  
25%                           -0.167747  
50%                            0.561642  
75%                            1.119316  
max                           13.103565  


## F. 돌풍(gust) 비율

GFS는 지표 돌풍(순간최대풍속) `surface_0_gust`를 직접 제공합니다. 평균풍속 대비 돌풍의 비율이 크면 그만큼 바람이 변덕스럽다는 뜻으로, 출력 변동성의 대리변수입니다(`wind-domain-features` 5절). 전 그룹 공유(GFS 격자 1개)로 계산합니다.


In [10]:
def add_gust_ratio(df, feats):
    gfs_gust = group_weighted_scalar(df, "kpx_group_1", "gfs", "surface_0_gust")
    ws10_floor = feats["gfs_ws10m"].clip(lower=WS_FLOOR)
    feats["gfs_gust_ratio"] = gfs_gust / ws10_floor
    return feats


train_feats = add_gust_ratio(train_clean, train_feats)
test_feats = add_gust_ratio(test_clean, test_feats)

print(train_feats["gfs_gust_ratio"].describe())


count    26304.000000
mean         1.709826
std          0.870357
min          0.408436
25%          1.043599
50%          1.504352
75%          2.189999
max         26.484293
Name: gfs_gust_ratio, dtype: float64


## G. NWP 앙상블 불일치 (GFS vs LDAPS)

서로 다른 두 예보 모델(GFS: 전지구·안정적, LDAPS: 고해상도·지형 반영 우수)이 같은 시각에 서로 다르게 예측한다면, 그 차이 자체가 "이번 시각은 예보 불확실성이 크다"는 신호가 될 수 있습니다(GEFCom2014 등 풍력 예측 대회 상위권의 공통 전략, `wind-domain-features` 5절). **같은 높이(10m)** 끼리 비교해야 모델 차이만 순수하게 볼 수 있어서, GFS 10m과 LDAPS 10m을 비교합니다(EDA의 GFS 100m vs SCADA 비교와는 다른 목적입니다). LDAPS는 그룹별로 다르므로 그룹마다 계산합니다.


In [11]:
def add_nwp_disagreement(feats):
    for group in TARGET_COLS:
        feats[f"{group}_nwp_ws10m_diff"] = feats["gfs_ws10m"] - feats[f"{group}_ldaps_ws10m"]
    return feats


train_feats = add_nwp_disagreement(train_feats)
test_feats = add_nwp_disagreement(test_feats)

print(train_feats[[f"{g}_nwp_ws10m_diff" for g in TARGET_COLS]].describe())


       kpx_group_1_nwp_ws10m_diff  kpx_group_2_nwp_ws10m_diff  \
count                26304.000000                26304.000000   
mean                    -2.355090                   -2.524277   
std                      1.519203                    1.545674   
min                    -10.692457                  -10.841388   
25%                     -3.201972                   -3.435380   
50%                     -2.165408                   -2.372606   
75%                     -1.320561                   -1.470529   
max                      4.472280                    5.067120   

       kpx_group_3_nwp_ws10m_diff  
count                26304.000000  
mean                    -2.644903  
std                      1.730837  
min                    -11.001748  
25%                     -3.731430  
50%                     -2.444304  
75%                     -1.399542  
max                      4.698403  


## H. 발표분 내 변화율

같은 발표분(`data_available_kst_dtm`) 안에서 이전 시각 대비 풍속이 얼마나 바뀌었는지(`diff`)는 전선 통과 같은 급변 상황을 포착할 수 있는 피처입니다. 같은 발표분 안에서만 비교하므로 발표분 경계를 넘지 않고, 이미 공개된 정보만 쓰는 셈이라 `leakage-guard` 기준으로 안전합니다("같은 발표분 내 diff ✅ 허용").

각 발표분의 **첫 시각은 비교할 이전 시각이 없어 자연스럽게 결측(NaN)**이 됩니다 — 누수가 아니라 그냥 정의상 값이 없는 것이며, 모델(LightGBM 등)은 NaN을 그대로 다룰 수 있습니다.


In [12]:
def add_issuance_diff(df, feats):
    gfs_avail = df["gfs_data_available_kst_dtm"].values
    feats["gfs_ws100m_diff_prev"] = feats.groupby(gfs_avail)["gfs_ws100m"].diff()

    ldaps_avail = df["ldaps_data_available_kst_dtm"].values
    for group in TARGET_COLS:
        feats[f"{group}_ldaps_ws10m_diff_prev"] = feats.groupby(ldaps_avail)[f"{group}_ldaps_ws10m"].diff()
    return feats


train_feats = add_issuance_diff(train_clean, train_feats)
test_feats = add_issuance_diff(test_clean, test_feats)

print("gfs_ws100m_diff_prev 결측(발표분 첫 시각) 개수:", train_feats["gfs_ws100m_diff_prev"].isna().sum())
print(train_feats["gfs_ws100m_diff_prev"].describe())


gfs_ws100m_diff_prev 결측(발표분 첫 시각) 개수: 1096
count    25208.000000
mean        -0.001993
std          0.742355
min         -7.738283
25%         -0.305242
50%         -0.004548
75%          0.290256
max          7.427299
Name: gfs_ws100m_diff_prev, dtype: float64


## I. 시간(달력) 피처

시간(hour)·달(month)은 원형 변수이므로(23시와 0시는 실제로는 1시간 차이) `sin`/`cos`로 인코딩합니다. 산악풍은 일주기(낮/밤)·계절 순환이 뚜렷하므로 트리 모델이 이 주기성과 풍속의 상호작용(예: 겨울×저풍속 유리, 겨울×고풍속 컷아웃 불리)을 스스로 분할해 학습할 수 있도록 재료를 제공하는 목적입니다. 계절 상호작용은 D절의 밀도 피처가 물리적 경로를 이미 상당 부분 설명하므로, 여기서는 별도의 "겨울 더미" 없이 최소한(hour/month sin·cos)만 추가합니다.


In [13]:
def add_time_features(df, feats):
    dt = df["forecast_kst_dtm"]
    feats["hour_sin"] = np.sin(2 * np.pi * dt.dt.hour / 24)
    feats["hour_cos"] = np.cos(2 * np.pi * dt.dt.hour / 24)
    feats["month_sin"] = np.sin(2 * np.pi * dt.dt.month / 12)
    feats["month_cos"] = np.cos(2 * np.pi * dt.dt.month / 12)
    return feats


train_feats = add_time_features(train_clean, train_feats)
test_feats = add_time_features(test_clean, test_feats)

print(train_feats.shape, test_feats.shape)
display(train_feats.head(3))


(26304, 52) (8760, 53)


,forecast_kst_dtm,gfs_ws10m,gfs_ws80m,gfs_ws100m,gfs_wd100m_sin,gfs_wd100m_cos,gfs_ws100m_sq,gfs_ws100m_cube,gfs_ws850hpa,kpx_group_1_ldaps_ws10m,...,kpx_group_2_nwp_ws10m_diff,kpx_group_3_nwp_ws10m_diff,gfs_ws100m_diff_prev,kpx_group_1_ldaps_ws10m_diff_prev,kpx_group_2_ldaps_ws10m_diff_prev,kpx_group_3_ldaps_ws10m_diff_prev,hour_sin,hour_cos,month_sin,month_cos
0,2022-01-01 01:00:00,2.516111,3.159457,3.176957,-0.985030,-0.172384,10.093054,32.065196,4.717845,5.341039,...,-3.603697,-4.010693,NaN,NaN,NaN,NaN,0.258819,0.965926,0.5,0.866025
1,2022-01-01 02:00:00,2.473222,3.056382,3.076688,-0.978158,-0.207862,9.466008,29.123953,4.277518,4.534232,...,-2.753886,-2.935238,-0.100269,-0.806807,-0.892700,-1.118344,0.500000,0.866025,0.5,0.866025
2,2022-01-01 03:00:00,2.507237,3.105247,3.131877,-0.961744,-0.273951,9.808655,30.719503,4.022996,4.326244,...,-2.461819,-2.718449,0.055189,-0.207988,-0.258051,-0.182773,0.707107,0.707107,0.5,0.866025


## 저장 전 마무리

1. **학습 타깃 붙이기**: train에는 정답값(kpx_group_1/2/3)을 그대로 붙입니다. 결측인 행도 그대로 둡니다(학습 시 그룹별로 결측 제외, 앞서 결정한 방침).
2. **최종 점검**: 행 개수가 원본과 같은지(순서·시각이 안 바뀌었는지), 새로 생긴 피처 개수, 결측 분포를 확인합니다.


In [14]:
for col in TARGET_COLS:
    train_feats[col] = train_clean[col].values

assert (train_feats["forecast_kst_dtm"].values == train_clean["forecast_kst_dtm"].values).all()
assert (test_feats["forecast_kst_dtm"].values == test_clean["forecast_kst_dtm"].values).all()
assert len(train_feats) == len(train_base_wide)
assert len(test_feats) == len(test_base_wide)

print("train_feats:", train_feats.shape, "/ test_feats:", test_feats.shape)
print("\
컬럼당 결측 개수 (0보다 큰 것만, train):")
na_counts = train_feats.isna().sum()
print(na_counts[na_counts > 0])


train_feats: (26304, 55) / test_feats: (8760, 53)
컬럼당 결측 개수 (0보다 큰 것만, train):
gfs_ws100m_diff_prev                 1096
kpx_group_1_ldaps_ws10m_diff_prev    1096
kpx_group_2_ldaps_ws10m_diff_prev    1096
kpx_group_3_ldaps_ws10m_diff_prev    1096
kpx_group_1                           104
kpx_group_2                           103
kpx_group_3                          8766
dtype: int64


## 저장

`data/processed/train_features_v1.parquet`, `test_features_v1.parquet`로 저장합니다. 버전(`v1`)을 붙여, 이후 피처를 추가·수정할 때 이전 버전과 구분되도록 합니다.


In [15]:
train_feats.to_parquet(PROCESSED_DIR / "train_features_v1.parquet", index=False)
test_feats.to_parquet(PROCESSED_DIR / "test_features_v1.parquet", index=False)

print("저장 완료")
print("train_features_v1.parquet:", train_feats.shape)
print("test_features_v1.parquet:", test_feats.shape)


저장 완료
train_features_v1.parquet: (26304, 55)
test_features_v1.parquet: (8760, 53)


## 요약

**이 노트북이 결정한 것**
- LDAPS 상대습도 100% 초과 클리핑, test LDAPS 결측 3시각 발표분 내 시간보간 적용(train/test 동일 함수)
- 물리적 근거(풍속 3제곱, 윈드시어, 공기밀도, 대기안정도, 돌풍, NWP 앙상블 불일치, 발표분 내 변화율, 시간 주기성) 기반 피처 9개 그룹 생성
- GFS는 전 그룹 공유 피처, LDAPS는 그룹별 격자가중 피처로 분리 설계

**다음 노트북으로 넘기는 것**
- `data/processed/train_features_v1.parquet`, `test_features_v1.parquet`
- `04_model_selection.ipynb`에서 이 피처들로 베이스라인(평균, 선형회귀, LightGBM 기본값)을 만들고, `timeseries-validation`의 train/valid 분리(2022~2023 train, 2024 valid)로 비교
- 라벨 결측 시각은 그룹별 학습 시 마스킹(제외)해서 사용 — 이 노트북에서는 그대로 붙여만 둠
- 피처 개수가 많아 다음 `feature-selection` 단계에서 중요도·상관 기준으로 정리 필요
